# Introducing Cats

Common abstractions for functional programming in **Scala**

## What is Cats?

![cats](https://typelevel.org/cats/img/cats2.png)

 - Library providing *abstractions* for functional programming in Scala
 - Foundation for an ecosystem of *pure*, *typeful* libraries
 - Why **cats**?
     - Concepts coming from *“category theory”*

## What Cats offers

 - range of type classes
 - set of extension methods 
 - general FP primitives
 - (among many other things)

**Purpose:** allow to write general and *extensible* FP code

## How to include Cats in a project?

Including in an `build.sbt` project:

```
ThisBuild / version := "1.1"
ThisBuild / scalaVersion := "3.3.7"
ThisBuild / organization := "ch.hevs"
  libraryDependencies ++= Seq(
    "org.typelevel" %% "cats-core" % "2.13.0" ,
    "org.scalatest" %% "scalatest" % "3.2.14" % Test
)
```

## Importing Cats in this notebook

In [1]:
import $ivy.`org.typelevel::cats-core:2.13.0`

Downloaded https://repo1.maven.org/maven2/org/typelevel/cats-core_3/2.13.0/cats-core_3-2.13.0.pom
Downloaded https://repo1.maven.org/maven2/org/typelevel/cats-kernel_3/2.13.0/cats-kernel_3-2.13.0.pom


import $ivy.$                                


## Using Type Classes

Cats relies heavily on Type Classes (remember?)

 - enable ad-hoc polymorphism, i.e., overloading
 - alternative to subtyping
 - parametric polymorphism + ad-hoc polymorphism.



Example: remeber `Taxable[A]`:

In [2]:
trait Taxable[A]:
  extension(a:A)
    def computeTax:Double

defined trait Taxable

In [3]:
case class Architect(name: String, income:Double)

defined class Architect

In [4]:
given Taxable[Architect] with
  extension(a:Architect) def computeTax = a.income * 0.3

defined object 

## Show

The `toString` method has several problems, e.g. can be called on anything, and do unexpected things:

In [5]:
34.5.toString

res5: String = "34.5"

In [6]:
(new Architect("jim",34)).toString

res6: String = "Architect(jim,34.0)"

In [7]:
(new {val a=34}).toString

res7: String = "ammonite.$sess.cmd7$$anon$1@7690c697"

**Show** provides an alternative to `toString`, only to apply to desired classes.

The signature of the `show` function is given as:
```
def show[A](f: A => String): Show[A]
```

We can use `Show` applying it to a given type:

In [8]:
import cats.Show

import cats.Show


In [9]:
val showInt = Show.apply[Int]
showInt.show(4)

showInt: Show[Int] = cats.Show$$$Lambda/0x00007f8258b2ec20@52101707
res9_1: String = "4"

In [10]:
Show[Int].show(4)

res10: String = "4"

We can use it for different types, which are of course checked:

In [10]:
Show[Boolean].show(4)

-- [E007] Type Mismatch Error: cmd11.sc:1:31 -----------------------------------
1 |val res11 = Show[Boolean].show(4)
  |                               ^
  |      Found:    (4 : Int)
  |      Required: Boolean
  |
  |      The following import might make progress towards fixing the problem:
  |
  |        import sourcecode.Text.generate
  |
  |
  | longer explanation available when compiling with `-explain`
Compilation Failed

In [11]:
Show[Boolean].show(true)

res11: String = "true"

Cats also has lots of syntax sugar helpers that make things even simpler:

In [12]:
import cats.syntax.show.toShow

import cats.syntax.show.toShow


And now we can use `show` as an extension method:

In [13]:
4.show

res13: String = "4"

In [14]:
false.show

res14: String = "false"

In [15]:
List(4,8,7).show

res15: String = "List(4, 8, 7)"

We can also define a custom behavior for show, for example for the `Boolean` type:

In [16]:
given Show[Boolean] with
  def show(b: Boolean): String = b match
    case true => "vrai"
    case false => "faux"

defined object 

In [17]:
false.show

res17: String = "faux"

A simpler syntax is also possible:

In [18]:
given Show[Double] = Show.show(d => s"$d*10^0")

given_Show_Double: Show[Double] = <given>

In [19]:
39.45.show

res19: String = "39.45*10^0"

In [20]:
39.4f.show

res20: String = "39.4"

Surely we can do it for other types as well:

In [21]:
case class Student(name:String, age:Int)

defined class Student

In [22]:
given Show[Student] = 
  Show.show(s => s"${s.name}, ${s.age} years old")

given_Show_Student: Show[Student] = <given>

In [23]:
val will = Student("will",21)
will.show

will: Student = Student(name = "will", age = 21)
res23_1: String = "will, 21 years old"

## Equal

Equality with a bit of type checking

Consider we want to test this equality:

In [23]:
4 == true

-- [E172] Type Error: cmd24.sc:1:12 --------------------------------------------
1 |val res24 = 4 == true
  |            ^^^^^^^^^
  |          Values of types Int and Boolean cannot be compared with == or !=
Compilation Failed

We cannot even compile it as the types do not match. But if a supertype is used then types can often be blurred, and we compare even when it makes no sense:

In [24]:
(4:Any) == true

res24: Boolean = false

For other types this can lead to useless comparisons. 

Consider these `Doctor` and `Nurse` classes:

In [25]:
case class Doctor(name:String)
case class Nurse(name:String)

defined class Doctor
defined class Nurse

In [26]:
val doris = Doctor("doris")
val doris2 = Doctor("Doris")
val nick = Nurse("nick")

doris: Doctor = Doctor(name = "doris")
doris2: Doctor = Doctor(name = "Doris")
nick: Nurse = Nurse(name = "nick")

Comparison between a doctor and nurse will always be false as the types are not even the same:

In [27]:
doris == nick

res27: Boolean = false

This can be even more problematic in lists. Consider a list of `Option`

In [28]:
val list:List[Any] = List(8,21,6).map(Option(_))

list: List[Any] = List(Some(value = 8), Some(value = 21), Some(value = 6))

Then when we try to do a filter, we might not get what we expect:

In [29]:
list filter (i => i == 6)

res29: List[Any] = List()

Same on a list of `Doctor` or `Nurse`:

In [30]:
val list2 = List(doris,nick).map(Option(_))

list2: List[Option[scala.collection.immutable.List[scala.Option[ammonite.$sess.cmd26.wrapper.cmd25.Doctor | ammonite.$sess.cmd26.wrapper.cmd25.Nurse]]]] = List(
  Some(value = Doctor(name = "doris")),
  Some(value = Nurse(name = "nick"))
)

In [31]:
list2.filter(p => p == nick)

res31: List[Option[scala.collection.immutable.List[scala.Option[ammonite.$sess.cmd26.wrapper.cmd25.Doctor | ammonite.$sess.cmd26.wrapper.cmd25.Nurse]]]] = List(
)

### Introducing Eq

The Cats `Eq` provides some stronger type equality:

In [32]:
import cats.Eq
Eq[Int].eqv(123,123)

import cats.Eq

res32_1: Boolean = true

Now we can have compile-time errors:

In [32]:
Eq[Int].eqv(123,"123")

-- [E007] Type Mismatch Error: cmd33.sc:1:28 -----------------------------------
1 |val res33 = Eq[Int].eqv(123,"123")
  |                            ^^^^^
  |Found:    ("123" : String)
  |Required: Int
  |
  |One of the following imports might make progress towards fixing the problem:
  |
  |  import os.Path.stringPathValidated
  |  import os.PathChunk.stringPathChunkValidated
  |  import os.RelPath.stringRelPathValidated
  |  import os.SubPath.stringSubPathValidated
  |  import sourcecode.Text.generate
  |
  |
  | longer explanation available when compiling with `-explain`
Compilation Failed

Syntax is not the simplest, but cats as always brings some syntax coating to make things easier:

In [33]:
import cats.syntax.eq._

123 === 123

import cats.syntax.eq._


res33_1: Boolean = true

In [34]:
123 =!= 234

res34: Boolean = true

### Custom equality

Now we can use `Eq` to define equality in different ways.

For example we can define it on lower-case name comparison:

In [35]:
given Eq[Doctor] = 
  Eq.instance[Doctor]{(a,b)=>a.name.toLowerCase === b.name.toLowerCase}

given_Eq_Doctor: Eq[Doctor] = <given>

Now we can compare two Doctors:

In [36]:
doris === doris2

res36: Boolean = true

Type is checked anyway if we try to compare with a `Nurse`:

In [36]:
doris === nick

-- [E007] Type Mismatch Error: cmd37.sc:1:22 -----------------------------------
1 |val res37 = doris === nick
  |                      ^^^^
  |Found:    (cmd37.this.cmd26.nick : ammonite.$sess.cmd26².wrapper.cmd25.Nurse)
  |Required: ammonite.$sess.cmd26².wrapper.cmd25.Doctor
  |
  |where:    cmd26  is a value in class cmd37
  |          cmd26² is a object in package ammonite.$sess
  |
  |
  |The following import might make progress towards fixing the problem:
  |
  |  import sourcecode.Text.generate
  |
  |
  | longer explanation available when compiling with `-explain`
Compilation Failed

### Scala strict equality

Without Cats there are some type safety equality utils in the Scala lib:

In [46]:
import scala.language.strictEquality

doris == nick

import scala.language.strictEquality


res46_1: Boolean = false

We can allow comparison between two types, but it has to be made explicit:

In [47]:
given CanEqual[Doctor,Nurse] = CanEqual.derived

given_CanEqual_Doctor_Nurse: CanEqual[Doctor, Nurse] = <given>

In [48]:
doris == nick

res48: Boolean = false

This works on one way only:

In [43]:
nick == doris

res43: Boolean = false

In [44]:
given CanEqual[Nurse,Doctor] = CanEqual.derived

given_CanEqual_Nurse_Doctor: CanEqual[Nurse, Doctor] = <given>

In [45]:
nick == doris

res45: Boolean = false